Earth–Moon candidate coverage and descriptor sensitivity

Development EDA following notebooks 15–16 and the
[spatial evaluation plan](../../EDA_PLAN.md), defined before scoring.
Whole sites stay together across sizes and lunar views. All current sites have
already been explored; this is not an untouched evaluation or a trained model.
Physical bands describe a scenario, not validated analog equivalence.

In [ ]:
import hashlib
import itertools
import json
import math
import sys
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import rasterio
from IPython.display import Image, Markdown, display
from matplotlib.figure import Figure
from rasterio.crs import CRS
from rasterio.windows import Window

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from inspection import check_input, file_hash, verify_run
from patch_analysis import measure_planarity
from retrieval_analysis import terrain_discrepancies

NOTEBOOK = ROOT / 'notebooks/exploration/17_earth_moon_candidate_eda.ipynb'
PLAN = ROOT / 'EDA_PLAN.md'
PREVIOUS = ROOT / 'notebooks/diagnostics/scale-eda-20261029T012946475712Z'
RUN_IDS = {'earth_terrain_landcover': '20261028T063142045472Z_9d6d3b',
           'lunar_terrain_views': '20261028T084443441751Z_1a794f'}
FEATURES = ['slope_median_deg', 'slope_p90_deg', 'central_relief_m',
            'plane_residual_fraction', 'local_sd_median_m']
SCENARIOS = ['baseline', 'with_planarity', 'with_local_sd']
SIZES = [300, 600, 960]
TOP_K = 5
OUT = ROOT / 'notebooks/diagnostics' / ('retrieval-eda-' + datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
evidence: dict[str, str] = {}
expected: dict[str, str] = {}
previous_manifest = json.loads((PREVIOUS / 'provenance.json').read_text(encoding='utf-8'))
assert previous_manifest['status'] == 'complete'
expected.update({str(PREVIOUS / name): digest for name, digest in previous_manifest['outputs'].items()})
evidence[str(PREVIOUS / 'provenance.json')] = file_hash(PREVIOUS / 'provenance.json')
all_metrics = pd.read_csv(check_input(PREVIOUS / 'center_metrics.csv', expected, evidence))
grids = json.loads(check_input(PREVIOUS / 'site_grids.json', expected, evidence).read_text())
units = pd.DataFrame(all_metrics.loc[all_metrics['common']]).copy()
units['spatial_group'] = units['body'] + ':' + units['region']
units['unit_id'] = units['center_id'] + ':' + units['size_m'].astype(str)
assert not units['unit_id'].duplicated().any()
assert np.isfinite(units[FEATURES].to_numpy(dtype=float)).all()
assert np.all(units.groupby('center_id').size().to_numpy() == 3)
assert set(units['population']) == {'natural_screen', 'full_context', 'image_supported'}
earth_sites = sorted(units.loc[units['body'] == 'earth', 'region'].unique())
moon_sites = sorted(units.loc[units['body'] == 'moon', 'region'].unique())
assert len(earth_sites) == 3 and len(moon_sites) == 7
runs = {stage: ROOT / 'data/processed' / stage / run for stage, run in RUN_IDS.items()}
for run in runs.values():
    expected.update(verify_run(run))
    evidence[str(run / 'provenance.json')] = file_hash(run / 'provenance.json')
evidence[str(PLAN)] = file_hash(PLAN)
source = '\n'.join(cell.source for cell in nbformat.read(NOTEBOOK, as_version=4).cells)
provenance: dict[str, Any] = {
    'status': 'running', 'notebook': str(NOTEBOOK.relative_to(ROOT)),
    'notebook_source_sha256': hashlib.sha256(source.encode()).hexdigest(),
    'helpers': {name: file_hash(ROOT / 'notebooks' / name) for name in ['inspection.py', 'patch_analysis.py', 'retrieval_analysis.py']},
    'runs': RUN_IDS, 'previous_eda': str(PREVIOUS.relative_to(ROOT)), 'sizes_m': SIZES,
    'cohort': 'notebook16 common 960 m centers, 960 m stride; separate lunar views',
    'group_policy': 'whole site across all scales, windows and views; all exploratory development',
    'context_allowance_m': 150, 'top_k': TOP_K, 'features': FEATURES,
    'score': 'maximum component discrepancy divided by fixed symmetric physical band',
    'bands': {'slope_median_deg': 1., 'slope_p90_deg': 2.,
              'central_relief_m': 'max(5, 0.10 * smaller relief)', 'plane_residual_fraction': .1,
              'local_sd_median_m': 'max(1, 0.10 * smaller local SD)'},
    'band_status': 'exploratory, not analog relevance labels or equipment limits',
    'tie_break': 'Earth site, center row, center column',
    'scenarios': SCENARIOS, 'pool_scenarios': ['all_sites', *[f'omit:{site}' for site in earth_sites]],
    'findings_led_thinning': {'site': 'atacama_san_pedro', 'size_m': 600, 'counts': [32, 701],
                             'seeds': list(range(20261029, 20261034)), 'scope': 'Atacama-only baseline'},
    'source_detail_check': 'selected lunar query footprints; native 5 m source vs 30 m mean; source count and best-image-resolution maps',
    'python': sys.version, 'software': {name: version(name) for name in ['numpy', 'pandas', 'rasterio', 'matplotlib', 'nbformat']},
}
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .2})


def save_figure(fig: Figure, name: str) -> None:
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    display(Image(filename=str(OUT / f'{name}.png')))
    plt.close(fig)


print('Output:', OUT)

# Spatial groups and the represented population

Export site groups before any candidate scoring. Earth bounds come from the
pinned processing manifest. Lunar footprints use the recorded common projected
CRS; footprints expanded by 150 m along both axes must remain disjoint.
Separation prevents shared neighborhoods, not broader spatial/source dependence.
The candidate inventory is intentionally the common-center cohort, not every
eligible location. Site omissions below test collection dependence, not accuracy.

In [ ]:
earth_config = json.loads((runs['earth_terrain_landcover'] / 'provenance.json').read_text())['config']['areas']
group_rows: list[dict[str, Any]] = []
bounds: dict[str, tuple[float, float, float, float]] = {}
for region, grid in grids.items():
    height, width = grid['shape']
    transform = grid['transform']
    left, top = transform[2], transform[5]
    bounds[region] = (left, top - 30 * height, left + 30 * width, top)
    group_rows.append({'body': grid['body'], 'region': region,
                       'spatial_group': f'{grid["body"]}:{region}', 'role': 'exploratory_development',
                       'prior_exposure': 'site and patch EDA, notebooks14-16',
                       'bounds_projected_m': bounds[region], 'crs_wkt': grid['crs_wkt'],
                       'earth_bbox_lon_lat': earth_config[region]['bbox'] if grid['body'] == 'earth' else None,
                       'patch_records': int((units['region'] == region).sum())})
for first, second in itertools.combinations(earth_sites, 2):
    a, b = earth_config[first]['bbox'], earth_config[second]['bbox']
    assert a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1]
gap_rows: list[dict[str, Any]] = []
for first, second in itertools.combinations(moon_sites, 2):
    assert grids[first]['crs_wkt'] == grids[second]['crs_wkt']
    a, b = bounds[first], bounds[second]
    dx, dy = max(a[0] - b[2], b[0] - a[2], 0), max(a[1] - b[3], b[1] - a[3], 0)
    gap = math.hypot(dx, dy)
    assert dx > 300 or dy > 300, 'Reconsider spatial grouping: expanded footprints may overlap'
    gap_rows.append({'first': first, 'second': second, 'grid_gap_m': gap, 'context_allowance_each_m': 150,
                     'gap_minus_two_allowances_m': gap - 300, 'x_gap_m': dx, 'y_gap_m': dy,
                     'expanded_boxes_disjoint': dx > 300 or dy > 300})
(OUT / 'spatial_groups.json').write_text(json.dumps(group_rows, indent=2), encoding='utf-8')
pd.DataFrame(gap_rows).to_csv(OUT / 'lunar_group_separation.csv', index=False)
units.to_csv(OUT / 'analysis_units.csv.gz', index=False, compression='gzip')
pool_definitions = [{'pool': 'all_sites', 'included_sites': earth_sites, 'role': 'development inventory'}]
pool_definitions += [{'pool': f'omit:{site}', 'included_sites': [r for r in earth_sites if r != site],
                      'role': 'candidate availability stress scenario, not training/test fold'} for site in earth_sites]
(OUT / 'candidate_pools.json').write_text(json.dumps(pool_definitions, indent=2), encoding='utf-8')
display(pd.DataFrame(group_rows)[['region', 'spatial_group', 'role', 'patch_records']])
print('Minimum lunar grid separation (m):', min(row['grid_gap_m'] for row in gap_rows))
site_colors = dict(zip(earth_sites, ['#bd650d', '#007b83', '#6c4cab'], strict=True))
fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout='constrained')
for row_index, population in enumerate(['full_context', 'image_supported']):
    for column, size in enumerate(SIZES):
        ax = axes[row_index, column]
        for site in earth_sites:
            earth = units[(units['region'] == site) & (units['size_m'] == size)]
            ax.scatter(earth['slope_p90_deg'], earth['central_relief_m'], s=4, alpha=.35, color=site_colors[site], label=site)
        moon = units[(units['population'] == population) & (units['size_m'] == size)]
        ax.scatter(moon['slope_p90_deg'], moon['central_relief_m'], s=5, alpha=.25, color='#333333', label='Moon')
        ax.set_title(f'{population} | {size} m')
        ax.set_xlabel('P90 slope (degrees)')
        ax.set_ylabel('Central relief (m; symlog)')
        ax.set_yscale('symlog', linthresh=5)
        if column == 0:
            ax.legend(fontsize=7)
fig.suptitle('Descriptor-space coverage in the common-center cohort\nOverlapping points and marginal overlap do not establish analog matches')
save_figure(fig, 'descriptor_space')

# Candidate comparisons with fixed physical bands

Source: natural-screened Earth windows. Metadata: descriptors, coordinates,
source support and group IDs. Filter: same size and selected site pool.
Score each pair independently with the maximum standardized discrepancy.
Select five smallest values with deterministic ties; preserve the physical
components and whether every band is met. Save artifacts synchronously for
reproducibility; no serving system or online side effects are implemented.

Baseline bands: median slope 1 degree, P90 slope 2 degrees, central relief
max(5 m, 10% of the smaller relief). Add either a 0.1 plane-fraction band or
local SD max(1 m, 10% of the smaller SD). No learned scaling or weights.
Added components can only raise this score. A closest result above 1 is an
out-of-band candidate, not evidence that the collection contains an analog.

In [ ]:
query_rows: list[dict[str, Any]] = []
ranking_rows: list[dict[str, Any]] = []
site_rows: list[dict[str, Any]] = []
turnover_rows: list[dict[str, Any]] = []
for size in SIZES:
    candidates = pd.DataFrame(units.loc[(units['body'] == 'earth') & (units['size_m'] == size)]).sort_values(['region', 'center_row', 'center_col']).reset_index(drop=True)
    queries = pd.DataFrame(units.loc[(units['body'] == 'moon') & (units['size_m'] == size)]).sort_values(['region', 'population', 'center_row', 'center_col'])
    values = candidates[FEATURES].to_numpy(dtype=float)
    identifiers = candidates['unit_id'].to_numpy()
    candidate_regions = candidates['region'].to_numpy()
    masks = {site: candidate_regions == site for site in earth_sites}
    for query in queries.to_dict('records'):
        scores = terrain_discrepancies(np.array([query[f] for f in FEATURES], dtype=float), values)
        identity = {'query_id': query['unit_id'], 'region': query['region'], 'population': query['population'],
                    'size_m': size, 'center_row': query['center_row'], 'center_col': query['center_col']}
        selected_indices = {}
        for scenario in SCENARIOS:
            score = scores[scenario]
            order = np.argsort(score, kind='stable')
            selected_indices[scenario] = order[:TOP_K]
            for rank, index in enumerate(order[:TOP_K], start=1):
                ranking_rows.append({**identity, 'scenario': scenario, 'rank': rank,
                                     'candidate_id': identifiers[index], 'candidate_region': candidate_regions[index],
                                     'score': float(score[index]), 'within_bands': bool(score[index] <= 1),
                                     'baseline_score': float(scores['baseline'][index])})
            for site in earth_sites:
                indices = np.flatnonzero(masks[site])
                best = indices[int(np.argmin(score[indices]))]
                site_rows.append({**identity, 'scenario': scenario, 'candidate_site': site,
                                  'candidates': len(indices), 'within_band_count': int((score[indices] <= 1).sum()),
                                  'best_score': float(score[best]), 'best_candidate_id': identifiers[best]})
            for definition in pool_definitions:
                allowed = np.isin(candidate_regions, definition['included_sites'])
                valid_order = order[allowed[order]]
                best = int(valid_order[0])
                query_rows.append({**identity, 'scenario': scenario, 'pool': definition['pool'],
                                   'candidates': int(allowed.sum()), 'within_band_count': int((score[allowed] <= 1).sum()),
                                   'best_score': float(score[best]), 'best_candidate_id': identifiers[best],
                                   'best_candidate_region': candidate_regions[best]})
        base_set = set(map(int, selected_indices['baseline']))
        for scenario in ['with_planarity', 'with_local_sd']:
            changed_set = set(map(int, selected_indices[scenario]))
            turnover_rows.append({**identity, 'scenario': scenario,
                                  'top1_changed': int(selected_indices['baseline'][0]) != int(selected_indices[scenario][0]),
                                  'top5_overlap_fraction': len(base_set & changed_set) / TOP_K})
    print('Compared size:', size, 'queries:', len(queries), 'Earth candidates:', len(candidates))
results = pd.DataFrame(query_rows)
rankings = pd.DataFrame(ranking_rows)
per_site = pd.DataFrame(site_rows)
turnover = pd.DataFrame(turnover_rows)
results.to_csv(OUT / 'query_results.csv.gz', index=False, compression='gzip')
rankings.to_csv(OUT / 'top5_candidates.csv.gz', index=False, compression='gzip')
per_site.to_csv(OUT / 'per_site_candidates.csv.gz', index=False, compression='gzip')
turnover.to_csv(OUT / 'descriptor_turnover.csv', index=False)

# Coverage, site dependence and candidate changes

Report within-site coverage to avoid treating one large lunar region as a whole
body. Pooled summaries weight each query equally, so large regions contribute
more. Per-Earth-site counts also expose collection imbalance. Top-five turnover
describes sensitivity, not quality. The two lunar views overlap; paired results
at shared coordinates are kept separate from differences in their populations.

In [ ]:
coverage_rows: list[dict[str, Any]] = []
for _, raw_group in results.groupby(['region', 'population', 'size_m', 'scenario', 'pool'], sort=True):
    group = pd.DataFrame(raw_group)
    identity = {name: group.iloc[0][name] for name in ['region', 'population', 'size_m', 'scenario', 'pool']}
    coverage_rows.append({**identity, 'queries': len(group), 'within_band_queries': int((group['within_band_count'] > 0).sum()),
                          'within_band_pct': 100 * float((group['within_band_count'] > 0).mean()),
                          'median_best_score': float(group['best_score'].median()), 'p95_best_score': float(group['best_score'].quantile(.95))})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUT / 'coverage.csv', index=False)
pooled_rows: list[dict[str, Any]] = []
for _, raw_group in results.groupby(['population', 'size_m', 'scenario', 'pool'], sort=True):
    group = pd.DataFrame(raw_group)
    pooled_rows.append({**{name: group.iloc[0][name] for name in ['population', 'size_m', 'scenario', 'pool']},
                        'queries': len(group), 'within_band_pct': 100 * float((group['within_band_count'] > 0).mean()),
                        'median_best_score': float(group['best_score'].median())})
pooled = pd.DataFrame(pooled_rows)
pooled.to_csv(OUT / 'pooled_coverage.csv', index=False)
all_pool = pd.DataFrame(results.loc[results['pool'] == 'all_sites'])
paired = pd.DataFrame(all_pool.loc[all_pool['population'] == 'full_context']).merge(
    pd.DataFrame(all_pool.loc[all_pool['population'] == 'image_supported']),
    on=['region', 'size_m', 'scenario', 'center_row', 'center_col'], suffixes=('_full', '_strict'), validate='one_to_one')
paired['best_score_delta'] = paired['best_score_strict'] - paired['best_score_full']
paired['best_id_changed'] = paired['best_candidate_id_full'] != paired['best_candidate_id_strict']
paired.to_csv(OUT / 'shared_view_queries.csv.gz', index=False, compression='gzip')
scenario_colors = {'baseline': '#007b83', 'with_planarity': '#bd650d', 'with_local_sd': '#6c4cab'}
short_names = [site.replace('A3CLR22_', '').replace('_', ' ') for site in moon_sites]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), layout='constrained')
for row_index, population in enumerate(['full_context', 'image_supported']):
    for column, size in enumerate(SIZES):
        ax = axes[row_index, column]
        for offset, scenario in enumerate(SCENARIOS):
            rows = pd.DataFrame(coverage.loc[(coverage['population'] == population) & (coverage['size_m'] == size)
                                            & (coverage['scenario'] == scenario) & (coverage['pool'] == 'all_sites')]).set_index('region').loc[moon_sites]
            ax.scatter(rows['within_band_pct'], np.arange(7) + (offset - 1) * .2, label=scenario,
                       color=scenario_colors[scenario], s=15)
        ax.set_xlim(-2, 102)
        ax.set_yticks(range(7), short_names, fontsize=7)
        ax.invert_yaxis()
        ax.set_title(f'{population} | {size} m')
        ax.set_xlabel('Queries with >=1 Earth candidate inside all scenario bands (%)')
        if column == 0:
            ax.legend(fontsize=6)
fig.suptitle('Exploratory descriptor-band coverage; all three Earth sites\nBand coverage is not analog relevance or retrieval accuracy')
save_figure(fig, 'coverage_by_site')
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
for ax, population in zip(axes, ['full_context', 'image_supported'], strict=True):
    for scenario in SCENARIOS:
        rows = pd.DataFrame(pooled.loc[(pooled['population'] == population) & (pooled['size_m'] == 600)
                                       & (pooled['scenario'] == scenario)]).set_index('pool').loc[[p['pool'] for p in pool_definitions]]
        ax.plot(range(4), rows['within_band_pct'], marker='o', color=scenario_colors[scenario], label=scenario)
    ax.set_xticks(range(4), ['All sites', *[f'Omit\n{s}' for s in earth_sites]], fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_title(population)
    ax.set_ylabel('Query-weighted band coverage (%)')
    ax.legend(fontsize=8)
fig.suptitle('600 m candidate-collection sensitivity; same queries and bands\nOmissions are availability scenarios, not training/test folds')
save_figure(fig, 'site_omission')
turnover_summary = pd.DataFrame(turnover.groupby(['population', 'size_m', 'scenario']).agg(
    top1_changed_fraction=('top1_changed', 'mean'), mean_top5_overlap=('top5_overlap_fraction', 'mean'))).reset_index()
turnover_summary.to_csv(OUT / 'turnover_summary.csv', index=False)
display(pooled.loc[pooled['pool'] == 'all_sites'])
display(turnover_summary)

# Findings-led follow-up: candidate quantity versus terrain composition

The first run showed 61.8% baseline coverage of strict lunar queries at 600 m,
falling to zero without Atacama. Atacama supplies 3,078 candidate centers versus
701 Arizona and 32 Lanzarote centers. Check whether quantity alone can explain
that dependence: take five seeded uniform permutations of Atacama, retain the
first 32 or 701 centers (nested subsets), and compare their Atacama-only baseline
coverage with each other site's full inventory. Keep queries and bands fixed.

This is a bounded subsampling sensitivity, not a confidence interval or a
quality evaluation. Thinning changes which landforms are represented as well
as count. It cannot remove source differences or the common-cohort selection.

In [ ]:
atacama = pd.DataFrame(units.loc[(units['region'] == 'atacama_san_pedro') & (units['size_m'] == 600)]).sort_values(['center_row', 'center_col']).reset_index(drop=True)
query_600 = pd.DataFrame(units.loc[(units['body'] == 'moon') & (units['size_m'] == 600)])
assert len(units[(units['region'] == 'lanzarote') & (units['size_m'] == 600)]) == 32
assert len(units[(units['region'] == 'northern_arizona') & (units['size_m'] == 600)]) == 701
subsets: dict[tuple[int, int], np.ndarray] = {}
subset_rows: list[dict[str, Any]] = []
for seed in range(20261029, 20261034):
    permutation = np.random.default_rng(seed).permutation(len(atacama))
    for count in [32, 701]:
        indices = np.sort(permutation[:count])
        subsets[seed, count] = indices
        subset_rows += [{'seed': seed, 'candidate_count': count, 'candidate_id': atacama.iloc[int(index)]['unit_id']}
                        for index in indices]
thin_rows: list[dict[str, Any]] = []
atacama_values = atacama[FEATURES].to_numpy(dtype=float)
for query in query_600.to_dict('records'):
    scores = terrain_discrepancies(np.array([query[f] for f in FEATURES], dtype=float), atacama_values)['baseline']
    for (seed, count), indices in subsets.items():
        best = int(indices[int(np.argmin(scores[indices]))])
        thin_rows.append({'query_id': query['unit_id'], 'region': query['region'], 'population': query['population'],
                          'seed': seed, 'candidate_count': count, 'within_band_count': int((scores[indices] <= 1).sum()),
                          'best_score': float(scores[best]), 'best_candidate_id': atacama.iloc[best]['unit_id']})
thinning = pd.DataFrame(thin_rows)
pd.DataFrame(subset_rows).to_csv(OUT / 'thinning_subsets.csv', index=False)
thinning.to_csv(OUT / 'thinning_queries.csv.gz', index=False, compression='gzip')
thin_summary_rows: list[dict[str, Any]] = []
for _, raw_group in thinning.groupby(['population', 'seed', 'candidate_count'], sort=True):
    group = pd.DataFrame(raw_group)
    thin_summary_rows.append({**{name: group.iloc[0][name] for name in ['population', 'seed', 'candidate_count']},
                              'queries': len(group), 'within_band_pct': 100 * float((group['within_band_count'] > 0).mean())})
thin_summary = pd.DataFrame(thin_summary_rows)
thin_summary.to_csv(OUT / 'thinning_coverage.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
for ax, population in zip(axes, ['full_context', 'image_supported'], strict=True):
    for position, count in enumerate([32, 701]):
        values = thin_summary.loc[(thin_summary['population'] == population) & (thin_summary['candidate_count'] == count), 'within_band_pct']
        ax.scatter(np.full(len(values), position), values, color=site_colors['atacama_san_pedro'], label='Atacama subsets' if position == 0 else None)
        site = 'lanzarote' if count == 32 else 'northern_arizona'
        comparison = per_site[(per_site['population'] == population) & (per_site['size_m'] == 600)
                              & (per_site['scenario'] == 'baseline') & (per_site['candidate_site'] == site)]
        fraction = 100 * float((comparison['within_band_count'] > 0).mean())
        ax.scatter(position, fraction, marker='x', s=80, color=site_colors[site], label=f'{site} full inventory')
    ax.set_xticks([0, 1], ['32 candidates', '701 candidates'])
    ax.set_ylim(-2, 102)
    ax.set_ylabel('Baseline queries with >=1 inside-band candidate (%)')
    ax.set_title(population)
    ax.legend(fontsize=7)
fig.suptitle('600 m count-matched candidate sensitivity\nFive seeded Atacama subsets; points are scenarios, not confidence intervals')
save_figure(fig, 'candidate_count_sensitivity')
display(thin_summary)

# Inspect selected candidates and missing counterparts

For each lunar site/view at 600 m, choose a query with baseline band coverage
and the lowest total top-five overlap across the two added descriptors; ties
resolve by row/column. If none has baseline coverage, show the smallest baseline
discrepancy instead. Also show the largest baseline discrepancy within each
view at 600 m as a missing-counterpart stress example. These are deliberately
selected examples, not representative validation labels.

Each figure shows the target, baseline best, best with planarity, and best with
local SD. Duplicate candidates remain visible. Height is centered per patch for
display; colors and 600 m axes are shared within each figure. Colors can vary
between figures. Hatching marks lunar cells outside the strict complete-feature
mask. Full-context lunar heights may be prior-dominated.

In [ ]:
examples: list[dict[str, Any]] = []
base_600 = pd.DataFrame(all_pool.loc[(all_pool['size_m'] == 600) & (all_pool['scenario'] == 'baseline')])
overlap = pd.Series(turnover.loc[turnover['size_m'] == 600].groupby('query_id')['top5_overlap_fraction'].sum())
for _, raw_group in base_600.groupby(['region', 'population'], sort=True):
    group = pd.DataFrame(raw_group).copy()
    group['overlap_sum'] = pd.Series(group['query_id']).map(overlap)
    covered = pd.DataFrame(group.loc[group['within_band_count'] > 0])
    if len(covered):
        chosen = covered.sort_values(['overlap_sum', 'center_row', 'center_col']).iloc[0]
        reason = 'covered_query_with_largest_top5_change'
    else:
        chosen = group.sort_values(['best_score', 'center_row', 'center_col']).iloc[0]
        reason = 'no_covered_query_closest_available'
    examples.append({**chosen.to_dict(), 'selection_reason': reason})
for _, raw_group in base_600.groupby('population', sort=True):
    chosen = pd.DataFrame(raw_group).sort_values(['best_score', 'region', 'center_row', 'center_col'], ascending=[False, True, True, True]).iloc[0]
    examples.append({**chosen.to_dict(), 'selection_reason': 'largest_baseline_discrepancy'})
pd.DataFrame(examples).to_csv(OUT / 'gallery_queries.csv', index=False)
lookup = units.set_index('unit_id')
best_rows = pd.DataFrame(rankings.loc[(rankings['size_m'] == 600) & (rankings['rank'] == 1)]).set_index(['query_id', 'scenario'])
layer_cache: dict[tuple[str, str, str], np.ndarray] = {}


def gallery_layer(region: str, population: str, name: str) -> np.ndarray:
    key = region, population, name
    if key not in layer_cache:
        if population == 'natural_screen':
            path = runs['earth_terrain_landcover'] / 'earth' / region / 'grid_30m' / f'{name}.tif'
        elif name == 'image_supported_complete_feature_mask':
            path = runs['lunar_terrain_views'] / 'moon' / region / 'quality' / f'{name}.tif'
        else:
            path = runs['lunar_terrain_views'] / 'moon' / region / population / f'{name}.tif'
        with rasterio.open(check_input(path, expected, evidence)) as dataset:
            layer_cache[key] = dataset.read(1, masked=True).astype('float64').filled(np.nan)
    return layer_cache[key]


gallery_rows: list[dict[str, Any]] = []
for example_index, example in enumerate(examples):
    query_id = example['query_id']
    panels = [('query', query_id, 0.)]
    for scenario in SCENARIOS:
        best = best_rows.loc[(query_id, scenario)]
        panels.append((scenario, best['candidate_id'], float(best['score'])))
    images = []
    elevation_limit = slope_limit = 0.
    for label, identifier, score in panels:
        patch = lookup.loc[identifier]
        row, col = int(patch['row']), int(patch['col'])
        window = np.s_[row:row + 20, col:col + 20]
        elevation = gallery_layer(str(patch['region']), str(patch['population']), 'elevation_m')[window]
        slope = gallery_layer(str(patch['region']), str(patch['population']), 'slope_deg')[window]
        centered = elevation - float(elevation.mean())
        unsupported = (gallery_layer(str(patch['region']), str(patch['population']), 'image_supported_complete_feature_mask')[window] != 1
                       if patch['body'] == 'moon' else np.zeros((20, 20), dtype=bool))
        elevation_limit = max(elevation_limit, float(np.abs(centered).max()))
        slope_limit = max(slope_limit, float(slope.max()))
        images.append((patch, label, score, centered, slope, unsupported))
        gallery_rows.append({**patch.to_dict(), 'unit_id': identifier, 'query_id': query_id,
                             'example_index': example_index, 'panel': label, 'score': score,
                             'strict_complete_fraction': float(1 - unsupported.mean()) if patch['body'] == 'moon' else np.nan})
    fig, axes = plt.subplots(2, 4, figsize=(15, 8), layout='constrained')
    elevation_image = slope_image = None
    for column, (patch, label, score, centered, slope, unsupported) in enumerate(images):
        elevation_image = axes[0, column].imshow(centered, extent=(0, 600, 0, 600), cmap='RdBu_r', vmin=-elevation_limit, vmax=elevation_limit)
        slope_image = axes[1, column].imshow(slope, extent=(0, 600, 0, 600), cmap='viridis', vmin=0, vmax=slope_limit)
        status = 'target' if label == 'query' else (f'D={score:.2f}; ' + ('inside bands' if score <= 1 else 'outside bands'))
        axes[0, column].set_title(f'{label} | {status}\n{str(patch["region"]).replace("A3CLR22_", "")}\n'
                                  f'slope med/P90 {patch["slope_median_deg"]:.1f}/{patch["slope_p90_deg"]:.1f} deg; relief {patch["central_relief_m"]:.1f} m', fontsize=8)
        axes[1, column].set_title(f'plane fraction {patch["plane_residual_fraction"]:.2f}; RMS {patch["plane_residual_rms_m"]:.1f} m\n'
                                  f'local SD {patch["local_sd_median_m"]:.1f} m', fontsize=8)
        for ax in axes[:, column]:
            if unsupported.any():
                positions = np.r_[0, np.arange(15, 600, 30), 600]
                ax.contourf(positions, positions, np.pad(unsupported[::-1], 1, mode='edge'), levels=[.5, 1.5], colors='none', hatches=['////'])
            ax.set_xticks([0, 300, 600])
            ax.set_yticks([0, 300, 600])
            ax.set_xlabel('Grid easting offset (m)')
            ax.set_ylabel('Grid northing offset (m)')
    assert elevation_image is not None and slope_image is not None
    elevation_axes: list[Any] = [axes[0, column] for column in range(4)]
    slope_axes: list[Any] = [axes[1, column] for column in range(4)]
    fig.colorbar(elevation_image, ax=elevation_axes, label='Height minus patch mean (m)', shrink=.75)
    fig.colorbar(slope_image, ax=slope_axes, label='Slope (degrees)', shrink=.75)
    fig.suptitle(f'{example["region"]} | {example["population"]}\n{example["selection_reason"]}; 600 m; '
                 'colors shared within figure; hatching = outside strict complete-feature mask', fontsize=10)
    save_figure(fig, f'candidates_{example_index:02d}')
pd.DataFrame(gallery_rows).to_csv(OUT / 'gallery_patches.csv', index=False)

# Targeted source-detail check for the selected lunar examples

The galleries suggest smoother lunar forms than some Earth candidates. Check
whether this is already present in the published 5 m elevation, rather than
attributing it automatically to our 30 m averaging. Trace each unique selected
lunar footprint to its native source. Compare per-patch plane residual RMS and
fraction before/after averaging, and independently reconstruct the 30 m means.

Read the associated illuminated-image counts and best-input-image-resolution
map using the pinned local README's definitions. Best input image resolution
is not DEM effective resolution or accuracy. Counts and source resolution do
not distinguish real smooth terrain from producer reconstruction/regularization.
This is a selected-example diagnostic, not a population-wide source audit.
The quality rasters have actual 5/15 m grids despite their 005 filenames.
Summarize native quality-cell centers inside each footprint without resampling;
this is a native-center fraction, distinct from the processed strict mask.

In [ ]:
native_run = ROOT / 'data/processed/lunar_terrain_environment/20261028T022855014566Z_cffea7'
native_manifest_path = native_run / 'provenance.json'
native_manifest = json.loads(native_manifest_path.read_text())
assert native_manifest['status'] == 'complete'
evidence[str(native_manifest_path)] = file_hash(native_manifest_path)
expected.update({str(ROOT / name): item['sha256'] for name, item in native_manifest['inputs'].items()})
source_rows: list[dict[str, Any]] = []
source_arrays: dict[str, tuple[np.ndarray, np.ndarray]] = {}
selected_targets = pd.DataFrame(units.loc[units['unit_id'].isin([entry['query_id'] for entry in examples])]).drop_duplicates(['region', 'center_row', 'center_col'])


def native_window(path: Path, left: float, top: float, require_alignment: bool = False) -> tuple[np.ndarray, float]:
    if str(path) not in evidence:
        check_input(path, expected, evidence)
    with rasterio.open(path) as dataset:
        assert dataset.count == 1 and dataset.scales == (1.,) and dataset.offsets == (0.,)
        spacing = dataset.transform.a
        assert spacing in (5., 15.) and dataset.transform.e == -spacing
        assert dataset.transform.b == dataset.transform.d == 0
        assert dataset.crs == CRS.from_wkt(grids[path.parent.name]['crs_wkt'])
        col, row = ~dataset.transform * (left, top)
        if require_alignment:
            assert spacing == 5 and abs(row - round(row)) < 1e-6 and abs(col - round(col)) < 1e-6
        side = round(600 / spacing)
        first_row, first_col = math.ceil(row - .5), math.ceil(col - .5)
        assert 0 <= first_row <= dataset.height - side and 0 <= first_col <= dataset.width - side
        window = Window.from_slices((first_row, first_row + side), (first_col, first_col + side))
        values = dataset.read(1, window=window, masked=True).astype('float64').filled(np.nan)
        assert values.shape == (side, side)
        return values, spacing


for patch in selected_targets.to_dict('records'):
    region = patch['region']
    folder = ROOT / 'data/raw/moon/sdem/regions' / region
    check_input(folder / 'README.txt', expected, evidence)
    dem_path, = folder.glob('*_GLDELEV_005.tif')
    count_path, = folder.glob('*_GLDMASK_005.tif')
    resolution_path, = folder.glob('*_GLDBRES_005.tif')
    transform = grids[region]['transform']
    left = transform[2] + 30 * int(patch['col'])
    top = transform[5] - 30 * int(patch['row'])
    heights, native_spacing = native_window(dem_path, left, top, require_alignment=True)
    counts, count_spacing = native_window(count_path, left, top)
    best_resolution, resolution_spacing = native_window(resolution_path, left, top)
    assert np.isfinite(heights).all()
    with rasterio.open(dem_path) as dataset:
        assert dataset.crs is not None and dataset.crs == CRS.from_wkt(grids[region]['crs_wkt'])
    row, col = int(patch['row']), int(patch['col'])
    coarse = gallery_layer(region, 'full_context', 'elevation_m')[row:row + 20, col:col + 20]
    direct_mean = heights.reshape(20, 6, 20, 6).mean(axis=(1, 3))
    maximum_error = float(np.max(np.abs(direct_mean - coarse)))
    assert maximum_error < .001
    native_metrics, coarse_metrics = measure_planarity(heights), measure_planarity(coarse)
    usable_resolution = best_resolution[np.isfinite(best_resolution) & (best_resolution > 0)]
    usable_count = counts[np.isfinite(counts) & (counts >= 0)]
    source_rows.append({'unit_id': patch['unit_id'], 'region': region, 'center_row': patch['center_row'],
                        'center_col': patch['center_col'], 'left_m': left, 'top_m': top,
                        'source_elevation': str(dem_path.relative_to(ROOT)), 'source_count': str(count_path.relative_to(ROOT)),
                        'source_best_resolution': str(resolution_path.relative_to(ROOT)),
                        'native_grid_m': native_spacing, 'processed_grid_m': 30,
                        'count_grid_m': count_spacing, 'best_image_resolution_grid_m': resolution_spacing,
                        'quality_sampling': 'native cell centers within footprint; no resampling',
                        'mean_reconstruction_max_error_m': maximum_error,
                        **{f'native_{name}': value for name, value in native_metrics.items()},
                        **{f'processed_{name}': value for name, value in coarse_metrics.items()},
                        'positive_image_count_fraction': float((np.isfinite(counts) & (counts > 0)).mean()),
                        'count_map_valid_fraction': float(np.isfinite(counts).mean()),
                        'image_count_median': float(np.median(usable_count)) if usable_count.size else np.nan,
                        'best_image_resolution_valid_fraction': usable_resolution.size / best_resolution.size,
                        'best_image_resolution_median_m': float(np.median(usable_resolution)) if usable_resolution.size else np.nan})
    source_arrays[patch['unit_id']] = (heights, coarse)
source_detail = pd.DataFrame(source_rows)
source_detail.to_csv(OUT / 'selected_source_detail.csv', index=False)
display(source_detail)

# Display the extreme-mismatch physical footprint once, preserving raw grid spacing.
stress = next(example for example in examples if example['selection_reason'] == 'largest_baseline_discrepancy')
stress_query = lookup.loc[stress['query_id']]
source_record = source_detail[(source_detail['region'] == stress_query['region'])
                              & (source_detail['center_row'] == stress_query['center_row'])
                              & (source_detail['center_col'] == stress_query['center_col'])].iloc[0]
native_height, coarse_height = source_arrays[source_record['unit_id']]
earth_best = lookup.loc[best_rows.loc[(stress['query_id'], 'baseline'), 'candidate_id']]
row, col = int(earth_best['row']), int(earth_best['col'])
earth_height = gallery_layer(str(earth_best['region']), 'natural_screen', 'elevation_m')[row:row + 20, col:col + 20]
residual_maps = []
for heights in [native_height, coarse_height, earth_height]:
    yy, xx = np.indices(heights.shape)
    design = np.column_stack((np.ones(heights.size), xx.ravel(), yy.ravel()))
    coefficients = np.linalg.lstsq(design, heights.ravel(), rcond=None)[0]
    residual_maps.append((heights.ravel() - design @ coefficients).reshape(heights.shape))
limit = max(float(np.abs(values).max()) for values in residual_maps)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), layout='constrained')
residual_image = None
for ax, residual, label in zip(axes, residual_maps, ['Lunar published 5 m grid', 'Lunar processed 30 m grid', 'Baseline Earth candidate, 30 m grid'], strict=True):
    residual_image = ax.imshow(residual, extent=(0, 600, 0, 600), cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(f'{label}\nPlane residual RMS {np.sqrt(np.mean(residual**2)):.2f} m', fontsize=9)
    ax.set_xlabel('Grid easting offset (m)')
    ax.set_ylabel('Grid northing offset (m)')
assert residual_image is not None
residual_axes: list[Any] = [axes[index] for index in range(3)]
fig.colorbar(residual_image, ax=residual_axes, label='Elevation minus fitted plane (m)', shrink=.8)
fig.suptitle('Selected largest baseline mismatch: native versus processed detail\nShared physical footprint and color scale; grid spacing does not establish effective resolution')
save_figure(fig, 'native_detail_stress_case')

# Findings and next questions

Coverage is conditional on this sparse common-center inventory, fixed bands,
descriptors and source support. Candidate changes do not measure relevance.
Decide the next investigation from the coverage and maps rather than promoting
a scale, score or model because it can return a ranking.

In [ ]:
# Read the Markdown notes below for the standalone report export.
report_notebook = json.loads((ROOT / 'notebooks/exploration/17_earth_moon_candidate_eda.ipynb').read_text(encoding="utf-8"))
report_notes = {entry["metadata"]["report_section"]: "".join(entry["source"])
                for entry in report_notebook["cells"]
                if entry["cell_type"] == "markdown" and "report_section" in entry["metadata"]}
lines = [
    '# Earth–Moon candidate coverage and descriptor sensitivity',
    '',
    '## Scope and evaluation status',
    '',
    f'- {len(units):,} common-center patch records across three scales and ten site groups. Lunar views overlap; records are not independent observations.',
    *report_notes['narrative-64e9bfa2-1'].split('\n'),
    f"- Minimum lunar grid separation is {min(row['grid_gap_m'] for row in gap_rows):,.0f} m, exceeding the two 150 m feature-context allowances. This does not establish broader statistical independence.",
    *report_notes['narrative-64e9bfa2-2'].split('\n'),
]
for population in ['full_context', 'image_supported']:
    for size in SIZES:
        group = pd.DataFrame(pooled.loc[(pooled['population'] == population) & (pooled['size_m'] == size) & (pooled['pool'] == 'all_sites')]).set_index('scenario')
        lines.append(f'| {population} | {size} | {int(group.iloc[0]["queries"]):,} | '
                     f'{group.loc["baseline", "within_band_pct"]:.1f} | {group.loc["with_planarity", "within_band_pct"]:.1f} | {group.loc["with_local_sd", "within_band_pct"]:.1f} |')
lines += [
    *report_notes['narrative-64e9bfa2-3'].split('\n'),
]
for definition in pool_definitions:
    group = pd.DataFrame(pooled.loc[(pooled['size_m'] == 600) & (pooled['scenario'] == 'baseline') & (pooled['pool'] == definition['pool'])]).set_index('population')
    lines.append(f'| {definition["pool"]} | {group.loc["full_context", "within_band_pct"]:.1f} | {group.loc["image_supported", "within_band_pct"]:.1f} |')
lines += [
    *report_notes['narrative-64e9bfa2-4'].split('\n'),
]
for population in ['full_context', 'image_supported']:
    for count, site in [(32, 'lanzarote'), (701, 'northern_arizona')]:
        values = thin_summary.loc[(thin_summary['population'] == population) & (thin_summary['candidate_count'] == count), 'within_band_pct']
        comparison = per_site[(per_site['population'] == population) & (per_site['size_m'] == 600)
                              & (per_site['scenario'] == 'baseline') & (per_site['candidate_site'] == site)]
        fraction = 100 * float((comparison['within_band_count'] > 0).mean())
        lines.append(f'| {population} | {count} | {values.min():.1f}–{values.max():.1f} | {site} | {fraction:.1f} |')
lines += [
    *report_notes['narrative-64e9bfa2-5'].split('\n'),
]
for record in pd.DataFrame(turnover_summary.loc[turnover_summary['size_m'] == 600]).to_dict('records'):
    lines.append(f'- {record["population"]}, {record["scenario"]}: top candidate changes for {100 * record["top1_changed_fraction"]:.1f}% of queries; mean top-five overlap with baseline is {100 * record["mean_top5_overlap"]:.1f}%.')
lines += [
    f"- Shared lunar coordinates across views: {len(paired):,} size/scenario records; maximum absolute best-score difference {paired['best_score_delta'].abs().max():.6g}; best candidate changes {int(paired['best_id_changed'].sum()):,}. Separate this paired comparison from different full/strict populations.",
    '',
    '## Targeted native-source check',
    '',
    f"- Traced {len(source_detail)} unique selected lunar footprints to published 5 m elevation and associated count/best-image-resolution maps. Maximum discrepancy between direct 6x6 source means and processed 30 m cells: {source_detail['mean_reconstruction_max_error_m'].max():.6g} m.",
    f"- The selected largest-mismatch lunar footprint has plane residual RMS {source_record['native_plane_residual_rms_m']:.3f} m on the published 5 m grid versus {source_record['processed_plane_residual_rms_m']:.3f} m at 30 m. Its native-center positive-image-count fraction is {100 * source_record['positive_image_count_fraction']:.1f}%. This near-planar structure is already present in the published source; our averaging alone does not explain it.",
    *report_notes['narrative-64e9bfa2-6'].split('\n'),
]
report = '\n'.join(lines) + '\n'
(OUT / 'REPORT.md').write_text(report, encoding='utf-8')
provenance['inputs'] = {str(Path(p).relative_to(ROOT)): digest for p, digest in evidence.items()}
provenance['outputs'] = {p.name: file_hash(p) for p in sorted(OUT.iterdir()) if p.is_file() and p.name != 'provenance.json'}
provenance['status'] = 'complete'
provenance['completed_utc'] = datetime.now(UTC).isoformat()
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
display(Markdown(report))
print('Complete:', OUT)

- Whole-site groups and the spatial evaluation plan were defined before scoring. All current sites had prior EDA exposure; this is development exploration, not an untouched evaluation.

- Candidates are natural-screened Earth patches at the same size as each lunar query. The common 960 m cohort preserves paired centers but excludes fragmented terrain. Thirty-meter spacing does not equalize effective resolution.

## Fixed descriptor scenarios

Baseline: median slope within 1 degree, P90 slope within 2 degrees, central relief within max(5 m, 10% of the smaller relief). The ordering uses the largest discrepancy divided by its band. A score <=1 means every included band is met.

Additional scenarios separately include plane residual fraction (band 0.1), or median local SD (band max(1 m, 10% of the smaller value)). These are exploratory scenarios, not validated equivalence margins or operational limits. No population normalization or learned model is fitted. Additional maximum-score components cannot increase coverage.

## Query-weighted coverage with all Earth sites

| Lunar view | Size (m) | Queries | Baseline inside bands (%) | + Planarity (%) | + Local SD (%) |
| --- | --- | --- | --- | --- | --- |


These summaries weight each query equally; larger regions contribute more. coverage.csv and coverage_by_site.png retain each lunar region separately. Band coverage is not analog quality, and overlap in two plotted descriptors is insufficient.

## Dependence on Earth collection: 600 m baseline

| Pool | Full-context coverage (%) | Image-supported coverage (%) |
| --- | --- | --- |


Omitting a site changes the candidate inventory only; it is not a model-validation fold. Per-site best candidates and qualifying counts are exported for every query/scenario.

## Findings-led count-matched follow-up

Atacama contributes 3,078 common centers versus 701 Arizona and 32 Lanzarote centers. Five seeded Atacama permutations were thinned to 32/701 nested subsets, preserving the same 600 m queries and baseline bands.

| Lunar view | Candidate count | Atacama subset coverage range (%) | Other site at same count | Other-site coverage (%) |
| --- | --- | --- | --- | --- |


These five draws are a sensitivity check, not a confidence interval. Differences at the same count show candidate quantity alone is insufficient to explain the observed collection dependence; terrain composition, source characteristics and cohort selection remain relevant.

## Candidate sensitivity at 600 m


- Best-image-resolution metadata and image counts describe input support, not effective DEM detail or accuracy. This selected-example check cannot decide between real smooth terrain and producer reconstruction/regularization. Source-level or independent elevation evidence is still needed.

- Quality layers were summarized at their actual 5/15 m native grids using centers inside each footprint; the 005 filenames do not establish their pixel size. These fractions are distinct from the processed strict-complete mask.

## Inspection and interpretation

- Galleries select covered queries with large candidate turnover, plus closest cases where no query meets the baseline bands and the largest-discrepancy cases. These are selected examples, not independent relevance judgments.
- Each gallery shows the query and the best candidate under all three scenarios, including outside-band candidates. Read slope, relief, residual RMS, fraction and local SD together; near-flat amplitude and source support can change their interpretation.
- Scalar summaries are incomplete shape representations. Candidate turnover motivates inspection but does not establish that the added descriptor improves retrieval. Orientation-dependent mission questions require geographic context.
- Selected lunar patches appear smoother than some Earth candidates despite similar slope/relief. Actual landform differences, effective resolution and DEM construction are competing explanations; this visual observation is not a source-quality diagnosis.
- Full-context lunar detail can be prior-dominated. Strict-mask hatching and separate image-supported results preserve this distinction. Natural Earth screening establishes neither access nor safety.

## Next questions

- Prioritize the smoother lunar/rougher Earth mismatch: inspect source support and effective detail before tuning feature weights or training a CNN. Then check whether the common 960 m requirement excludes useful 600 m Earth candidates.
- Define independent analog-relevance criteria and spatial evaluation data before tuning descriptor bands or training a CNN. Current site groups prevent neighborhood mixing but cannot erase prior exploratory exposure.
- If candidate morphology is understandable, associate environmental evidence at its native support in a separate cycle; keep terrain resemblance and operational suitability distinct.

## Artifacts

- spatial_groups.json, lunar_group_separation.csv and candidate_pools.json: groups and explicit development scenarios.
- analysis_units.csv.gz: the exact georeferenced common-center population and descriptors.
- query_results.csv.gz: every query/descriptor/pool result; per_site_candidates.csv.gz: candidate counts and best per Earth site.
- top5_candidates.csv.gz: deterministic full-pool rankings, scores, band status and baseline discrepancy.
- coverage.csv, pooled_coverage.csv, descriptor_turnover.csv and turnover_summary.csv: descriptive coverage and sensitivity.
- shared_view_queries.csv.gz: shared-coordinate full/strict comparison.
- thinning_subsets.csv, thinning_queries.csv.gz and thinning_coverage.csv: exact seeded count-matched sensitivity.
- selected_source_detail.csv and native_detail_stress_case.png/pdf: traced native footprints, reconstruction checks and residual maps.
- gallery_queries.csv, gallery_patches.csv and PNG/PDF figures: deterministic selections, physical context and support.
- provenance.json: pinned input hashes, source code, configuration, software and output hashes. Raw and processed datasets are unchanged.